### Linear integer program

Let $I = \{0,\dots,\text{num\_ambulances}-1\}$ and $J = \{0,\dots,|\text{ambulance\_posts}|-1\}$ and $K = \{0,\dots,|\text{call\_hotspots}|-1\}$.

Decision variable:

- $x_{ij} \in \{0,1\}$, equal to 1 if ambulance $i$ is assigned to post $j$.
- $y_{k} \in \{0,1\}$, equal to 1 if call hotspot $k$ is "covered" by at least one ambulance (this is enforced by constraints).

Parameters:
- $c_{ij}$ cost of ambulanc $i$ moving to post $j$ (distance travelled but shift change cost)
- $b_{k}$ intensity of call hotspot (for example average number of calls during time period)

Objective:

$$
\min \sum_{i \in I} \sum_{j \in J} (c_{ij}) x_{ij} - \sum_{k \in K} (b_k) y_k
$$

Constraints

Each ambulance assigned once:

$$
\sum_{j \in J} x_{ij} = 1, \quad \forall i \in I
$$

Call hotspot covered constraint, where $J^{(*k)} \subseteq J$ is the subset $\{j \in J \text{ given ambulance post } j \text{ covers call hotspot k}  \}$:

$$
\sum_{j \in J^{(*k)}} x_{ij} \geq y_k, \quad \forall k \in K
$$


<!-- Optional post capacity / load balancing requirement constraint:

$$
\sum_{i \in I} x_{ij} \le U_j, \quad \forall j \in J
$$ -->

Possible to denote diminisihing award for coverage. For example assume $z_k$ is either 1 or 0. This constraint ensures it is $1$ only when there are at least 2 ambulances in the coverage spot:

$$
\sum_{j \in J^{(*k)}} x_{ij} \geq z_k - 1, \quad \forall k \in K
$$


In [1]:
# Integer Program
# ambulance variables is x_ij for i in num_ambulance and j in ambulance_posts
# c_j is cost of being at ambulance post j, which is the distance to the nearest hospital

import pulp


def build_ambulance_integer_program(num_ambulances, ambulance_posts, c, b, J_k, post_capacity=None):
    I = range(num_ambulances)
    J = range(len(ambulance_posts))
    K = range(len(b))

    model = pulp.LpProblem("AmbulancePostAssignment", pulp.LpMinimize)
    x = pulp.LpVariable.dicts("x", (I, J), lowBound=0, upBound=1, cat="Binary")
    y = pulp.LpVariable.dicts("y", K, lowBound=0, upBound=1, cat="Binary")

    model += (
        pulp.lpSum(c[j] * x[i][j] for i in I for j in J)
        - pulp.lpSum(b[k] * y[k] for k in K)
    )

    for i in I:
        model += pulp.lpSum(x[i][j] for j in J) == 1

    for k in K:
        covering_posts = J_k.get(k, [])
        model += pulp.lpSum(x[i][j] for i in I for j in covering_posts) >= y[k]

    if post_capacity is not None:
        for j in J:
            model += pulp.lpSum(x[i][j] for i in I) <= post_capacity[j]

    return model, x, y

In [2]:
hosptials = [(0,0), (0,10), (10,10), (10,0)]
ambulance_posts = [(0, 3), (10, 7), (1, 10)]
num_ambulances = 5

In [3]:
import math

hospitals = hosptials


def euclidean_distance(a, b):
    return math.dist(a, b)


c = [
    min(euclidean_distance(post, hospital) for hospital in hospitals)
    for post in ambulance_posts
]

c = [
    3, 1000, 1
]

call_hotspots = [(0, 4), (9, 8), (2, 10)]

b = [8, 6, 10]

# J_k[k] = list of post indices j that can cover hotspot k
J_k = {
    0: [0],
    1: [1],
    2: [2],
}

model, x, y = build_ambulance_integer_program(num_ambulances, ambulance_posts, c, b, J_k)
print("c =", c)
print("b =", b)
print("J_k =", J_k)

c = [3, 1000, 1]
b = [8, 6, 10]
J_k = {0: [0], 1: [1], 2: [2]}


In [4]:
x

{0: {0: x_0_0, 1: x_0_1, 2: x_0_2},
 1: {0: x_1_0, 1: x_1_1, 2: x_1_2},
 2: {0: x_2_0, 1: x_2_1, 2: x_2_2},
 3: {0: x_3_0, 1: x_3_1, 2: x_3_2},
 4: {0: x_4_0, 1: x_4_1, 2: x_4_2}}

In [5]:
status = model.solve(pulp.PULP_CBC_CMD(msg=False))

print("Status:", pulp.LpStatus[status])
print("Objective:", pulp.value(model.objective))

for i in range(num_ambulances):
    for j in range(len(ambulance_posts)):
        if pulp.value(x[i][j]) > 0.5:
            print(f"Ambulance {i} -> Post {j} at {ambulance_posts[j]}")

covered_hotspots = [k for k in range(len(b)) if pulp.value(y[k]) > 0.5]
print("Covered hotspots:", covered_hotspots)

Status: Optimal
Objective: -11.0
Ambulance 0 -> Post 2 at (1, 10)
Ambulance 1 -> Post 2 at (1, 10)
Ambulance 2 -> Post 0 at (0, 3)
Ambulance 3 -> Post 2 at (1, 10)
Ambulance 4 -> Post 2 at (1, 10)
Covered hotspots: [0, 2]
